# LSTM Short Text Classification — Experimentation Notebook

> **Đề tài**: Xây dựng mô hình học sâu LSTM cho bài toán phân loại văn bản ngắn tiếng Việt

Notebook này phục vụ cho:
- Khám phá dữ liệu (EDA)
- Thử nghiệm nhanh các ý tưởng
- Phân tích kết quả và visualisation

**Logic chính được tách ra `src/`** — notebook chỉ là giao diện thực nghiệm.

In [ ]:
import sys
from pathlib import Path

# Thêm project root vào path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

## 1. Khám phá dữ liệu (EDA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import CFG

# Load raw data
df = pd.read_excel(CFG.data.raw_path)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Phân phối nhãn
label_col = CFG.data.label_col
fig, ax = plt.subplots(figsize=(8, 4))
df[label_col].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Phân phối nhãn cảm xúc', fontsize=13)
ax.set_xlabel('Nhãn')
ax.set_ylabel('Số mẫu')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Độ dài văn bản
df['char_len'] = df[CFG.data.text_col].str.len()
df['word_len'] = df[CFG.data.text_col].str.split().str.len()

print(df[['char_len', 'word_len']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['char_len'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Phân phối độ dài văn bản (ký tự)')
df['word_len'].hist(bins=50, ax=axes[1], color='orange', edgecolor='white')
axes[1].set_title('Phân phối số từ')
for ax in axes:
    ax.set_xlabel('Độ dài')
    ax.set_ylabel('Tần suất')
plt.tight_layout()
plt.show()

## 2. Thử nghiệm Preprocessing

In [ ]:
from src.preprocessing.clean_text import clean_text
from src.preprocessing.teencode_normalize import normalize_teencode

examples = [
    'Hôm nay mk dc gặp crush!! Vui vcl haha 😍😍',
    'Ko hiểu sao mik lại sad z... huhu',
    'Tức quá wtf! Ng ta lm j z mà tui phải chịu',
    'ok nhe! Thx bn nhiu lắm nha 🙏',
]

for text in examples:
    cleaned = clean_text(text, remove_emoji=False)
    normed  = normalize_teencode(cleaned)
    print(f'RAW    : {text}')
    print(f'CLEANED: {cleaned}')
    print(f'NORMED : {normed}')
    print('-' * 60)

## 3. Kiểm tra Vocabulary

In [ ]:
from pathlib import Path
from src.preprocessing.vocabulary import Vocabulary

vocab_path = Path(CFG.data.processed_dir) / 'vocabulary.pkl'
if vocab_path.exists():
    vocab = Vocabulary.load(vocab_path)
    print(f'Vocabulary size: {vocab.size}')
    print(f'PAD idx: {vocab.PAD_IDX}')
    print(f'UNK idx: {vocab.UNK_IDX}')
    
    # Test encoding
    text = 'tôi rất vui hôm nay'
    ids  = vocab.encode(text, max_len=20)
    decoded = vocab.decode(ids)
    print(f'\nText  : {text}')
    print(f'IDs   : {ids}')
    print(f'Decode: {decoded}')
else:
    print('Vocabulary not found. Run: python main.py preprocess')

## 4. Kiến trúc mô hình BiLSTM

In [ ]:
from src.models.lstm_model import BiLSTMClassifier

# Tạo model với config mặc định
cfg_m = CFG.models.lstm
model = BiLSTMClassifier(
    vocab_size=15000,   # dummy size for inspection
    embed_dim=cfg_m.embed_dim,
    hidden_size=cfg_m.hidden_size,
    num_layers=cfg_m.num_layers,
    num_classes=CFG.labels.num_classes,
    dropout=cfg_m.dropout,
    bidirectional=cfg_m.bidirectional,
    use_attention=cfg_m.use_attention,
)
print(model)
print(f'\nTrainable parameters: {model.count_parameters():,}')

## 5. Phân tích Attention Weights

In [ ]:
from pathlib import Path

model_path = Path(CFG.outputs.models_dir) / 'lstm_best.pt'
if model_path.exists():
    from src.inference.predict import LSTMPredictor
    predictor = LSTMPredictor()
    
    test_texts = [
        'Hôm nay tôi rất vui được gặp bạn bè!!',
        'Tôi buồn quá, không muốn làm gì cả',
        'Trời ơi tức quá đi! Sao lại như vậy được',
    ]
    
    for text in test_texts:
        result  = predictor.predict(text)
        weights = predictor.get_attention_weights(text)
        
        print(f'Text    : {text}')
        print(f'Emotion : {result["predicted_name"]} ({result["confidence"]:.1%})')
        if weights:
            top3 = sorted(weights.items(), key=lambda x: -x[1])[:3]
            print(f'Top-3 attention: {top3}')
        print('-' * 60)
else:
    print('BiLSTM model not trained yet. Run: python main.py train lstm')

## 6. Kết quả so sánh mô hình

In [ ]:
import json

report_path = Path(CFG.outputs.reports_dir) / 'comparison.json'
if report_path.exists():
    with open(report_path) as f:
        results = json.load(f)
    
    # Summary table
    rows = []
    for name, m in results.items():
        rows.append({
            'Model': name,
            'Accuracy': m.get('accuracy', '-'),
            'Precision': m.get('precision', '-'),
            'Recall': m.get('recall', '-'),
            'F1': m.get('f1', '-'),
            'Params': f"{m.get('num_params', 0):,}",
            'Inf. (ms)': m.get('inference_ms_per_sample', '-'),
        })
    
    summary = pd.DataFrame(rows).set_index('Model')
    print(summary.to_string())
else:
    print('No comparison report found. Run: python main.py evaluate')

In [ ]:
# Training curves
from src.evaluation.visualization import plot_training_curves

for model_name, fname in [('BiLSTM+Attention', 'lstm_history.json'),
                            ('DNN+TF-IDF', 'dnn_history.json')]:
    hist_path = Path(CFG.outputs.logs_dir) / fname
    if hist_path.exists():
        with open(hist_path) as f:
            history = json.load(f)
        plot_training_curves(history, model_name)
        plt.show()
    else:
        print(f'{model_name} history not found')